# Week 2 lab — how much service capacity prevents waiting?

**Driving question.** In the stated 09:00–11:00 service scenario, where is the critical staffing range, and how much does adding a seventeenth agent change waiting relative to sixteen agents?

This notebook follows the course loop:

`observe arrivals → build a queue model → simulate → check → compare staffing`

The arrival extract counts entries to the automated **VRU**, not arrivals to a human-agent queue. For this classroom scenario we assume that every selected VRU entry requests human service, service times are independent exponential variables with mean 6 minutes, agents are identical, service is FIFO, and callers do not abandon. Results are conditional on those assumptions; they are not a staffing recommendation for the bank.

**Work in your ignored copy.** From the repository root, copy this notebook before editing it:

```bash
mkdir -p students
cp notebooks/lecture2_queueing.ipynb students/lecture2_queueing_work.ipynb
```

The `students/` directory is ignored by Git. Never force-add, commit, or push this working copy.

Then open the copy with the same local Python environment used in Week 1. Start Jupyter or VS Code from the repository root so the data loader can find `data/`. The notebook uses only Python's standard library. See the [AnonymousBank data guide](../data/anonymousbank_data_guide.md) for the call journey and field definitions.


In [1]:
import csv
import math
import random
import statistics
from collections import defaultdict
from datetime import datetime
from pathlib import Path


## 1. Build the queue from a hand trace

For customer $i$:

- $A_i$: arrival time in minutes;
- $W_i$: waiting time;
- $B_i$: service-start time;
- $S_i$: service duration;
- $D_i$: departure time.

Use the five customers from the lecture. First calculate the one-agent and two-agent schedules by hand. With two agents, break a tie by choosing the lower agent number.


In [2]:
hand_arrivals = [0.0, 1.0, 2.5, 2.7, 3.0]
hand_services = [2.0, 2.0, 1.0, 2.5, 1.5]

print(f"{'Customer':>8} {'Arrival':>8} {'Service':>8}")
for customer, (arrival, service) in enumerate(
    zip(hand_arrivals, hand_services), start=1
):
    print(f"{customer:>8} {arrival:>8.1f} {service:>8.1f}")


Customer  Arrival  Service
       1      0.0      2.0
       2      1.0      2.0
       3      2.5      1.0
       4      2.7      2.5
       5      3.0      1.5


**Task 1. Implement the queue update.** Complete the marked block below.

Process customers in arrival order. `agent_free[j]` is the next time agent `j` becomes available. For each customer:

1. choose the agent with the earliest next-free time;
2. set the start time to the later of the arrival and that next-free time;
3. calculate departure and waiting time;
4. update that agent's next-free time;
5. append the record.


In [3]:
def simulate_queue(arrivals, services, number_of_agents):
    # Return one record per customer in arrival order.
    if not isinstance(number_of_agents, int) or number_of_agents <= 0:
        raise ValueError("number_of_agents must be a positive integer")
    if len(arrivals) != len(services):
        raise ValueError("arrivals and services must have equal length")
    if any(arrival < 0 for arrival in arrivals):
        raise ValueError("arrival times must be non-negative")
    if any(service <= 0 for service in services):
        raise ValueError("service times must be positive")
    if any(later < earlier for earlier, later in zip(arrivals, arrivals[1:])):
        raise ValueError("arrivals must be sorted")

    agent_free = [0.0] * number_of_agents
    records = []

    for customer, (arrival, service) in enumerate(
        zip(arrivals, services), start=1
    ):
        agent_index = min(range(number_of_agents), key=lambda j: agent_free[j])
        start = max(arrival, agent_free[agent_index])
        wait = start - arrival
        departure = start + service
        agent_free[agent_index] = departure

        records.append(
            {
                "customer": customer,
                "arrival": arrival,
                "wait": wait,
                "start": start,
                "service": service,
                "departure": departure,
                "agent": agent_index + 1,
            }
        )

    return records


In [5]:
def print_records(records, title=""):
    if title:
        print(title)
    print(f"{'Client':>6} {'Arrivée':>8} {'Début':>7} {'Attente':>8} {'Service':>8} {'Départ':>7} {'Agent':>6}")
    for r in records:
        print(
            f"{r['customer']:>6} {r['arrival']:>8.1f} {r['start']:>7.1f} "
            f"{r['wait']:>8.1f} {r['service']:>8.1f} {r['departure']:>7.1f} {r['agent']:>6}"
        )

print_records(simulate_queue(hand_arrivals, hand_services, 1), "=== 1 agent ===")
print()
print_records(simulate_queue(hand_arrivals, hand_services, 2), "=== 2 agents ===")

=== 1 agent ===
Client  Arrivée   Début  Attente  Service  Départ  Agent
     1      0.0     0.0      0.0      2.0     2.0      1
     2      1.0     2.0      1.0      2.0     4.0      1
     3      2.5     4.0      1.5      1.0     5.0      1
     4      2.7     5.0      2.3      2.5     7.5      1
     5      3.0     7.5      4.5      1.5     9.0      1

=== 2 agents ===
Client  Arrivée   Début  Attente  Service  Départ  Agent
     1      0.0     0.0      0.0      2.0     2.0      1
     2      1.0     1.0      0.0      2.0     3.0      2
     3      2.5     2.5      0.0      1.0     3.5      1
     4      2.7     3.0      0.3      2.5     5.5      2
     5      3.0     3.5      0.5      1.5     5.0      1


In [8]:
def trace_queue(arrivals, services, number_of_agents):
    agent_free = [0.0] * number_of_agents
    for customer, (arrival, service) in enumerate(zip(arrivals, services), start=1):
        before = agent_free.copy()
        agent_index = min(range(number_of_agents), key=lambda j: agent_free[j])
        start = max(arrival, agent_free[agent_index])
        wait = start - arrival
        departure = start + service
        agent_free[agent_index] = departure
        print(f"Client {customer} (arrivée {arrival}, service {service})")
        print(f"   agent_free avant : {before}")
        print(f"   -> agent {agent_index + 1} | start = max({arrival}, {before[agent_index]}) = {start}")
        print(f"   -> wait = {wait:.1f} | departure = {departure}")
        print(f"   agent_free après : {agent_free}")

trace_queue(hand_arrivals, hand_services, 2)

Client 1 (arrivée 0.0, service 2.0)
   agent_free avant : [0.0, 0.0]
   -> agent 1 | start = max(0.0, 0.0) = 0.0
   -> wait = 0.0 | departure = 2.0
   agent_free après : [2.0, 0.0]
Client 2 (arrivée 1.0, service 2.0)
   agent_free avant : [2.0, 0.0]
   -> agent 2 | start = max(1.0, 0.0) = 1.0
   -> wait = 0.0 | departure = 3.0
   agent_free après : [2.0, 3.0]
Client 3 (arrivée 2.5, service 1.0)
   agent_free avant : [2.0, 3.0]
   -> agent 1 | start = max(2.5, 2.0) = 2.5
   -> wait = 0.0 | departure = 3.5
   agent_free après : [3.5, 3.0]
Client 4 (arrivée 2.7, service 2.5)
   agent_free avant : [3.5, 3.0]
   -> agent 2 | start = max(2.7, 3.0) = 3.0
   -> wait = 0.3 | departure = 5.5
   agent_free après : [3.5, 5.5]
Client 5 (arrivée 3.0, service 1.5)
   agent_free avant : [3.5, 5.5]
   -> agent 1 | start = max(3.0, 3.5) = 3.5
   -> wait = 0.5 | departure = 5.0
   agent_free après : [5.0, 5.5]


In [9]:
for c in (1, 2, 3, 5):
    waits = [r["wait"] for r in simulate_queue(hand_arrivals, hand_services, c)]
    print(f"{c} agent(s) -> attentes {[round(w, 1) for w in waits]} | attente moyenne = {sum(waits)/len(waits):.2f}")

1 agent(s) -> attentes [0.0, 1.0, 1.5, 2.3, 4.5] | attente moyenne = 1.86
2 agent(s) -> attentes [0.0, 0.0, 0.0, 0.3, 0.5] | attente moyenne = 0.16
3 agent(s) -> attentes [0.0, 0.0, 0.0, 0.0, 0.0] | attente moyenne = 0.00
5 agent(s) -> attentes [0.0, 0.0, 0.0, 0.0, 0.0] | attente moyenne = 0.00


## 2. Check the implementation and keep the checks

A unit test is a small automatic check whose correct answer is already known. Run the code first, then verify each resulting schedule against your hand calculation. Passing these tests supports the implementation; it does not prove that every case is correct or that the model represents the bank.


In [4]:
def assert_close_list(observed, expected, tolerance=1e-9):
    assert len(observed) == len(expected)
    for got, wanted in zip(observed, expected):
        assert math.isclose(got, wanted, abs_tol=tolerance)

def test_hand_traces():
    one_agent = simulate_queue(hand_arrivals, hand_services, 1)
    two_agents = simulate_queue(hand_arrivals, hand_services, 2)

    assert_close_list(
        [record["wait"] for record in one_agent],
        [0.0, 1.0, 1.5, 2.3, 4.5],
    )
    assert_close_list(
        [record["wait"] for record in two_agents],
        [0.0, 0.0, 0.0, 0.3, 0.5],
    )
    assert [record["agent"] for record in two_agents] == [1, 2, 1, 2, 1]

    for record in one_agent + two_agents:
        assert record["start"] >= record["arrival"]
        assert record["wait"] >= 0
        assert math.isclose(
            record["departure"],
            record["start"] + record["service"],
        )

test_hand_traces()
print("Hand-trace tests passed.")


Hand-trace tests passed.


In [10]:
one_agent = simulate_queue(hand_arrivals, hand_services, 1)
two_agents = simulate_queue(hand_arrivals, hand_services, 2)
print("1 agent  attendues :", [0.0, 1.0, 1.5, 2.3, 4.5])
print("1 agent  obtenues  :", [round(r["wait"], 1) for r in one_agent])
print("2 agents attendues :", [0.0, 0.0, 0.0, 0.3, 0.5])
print("2 agents obtenues  :", [round(r["wait"], 1) for r in two_agents])
print("agents attendus    :", [1, 2, 1, 2, 1])
print("agents obtenus     :", [r["agent"] for r in two_agents])

1 agent  attendues : [0.0, 1.0, 1.5, 2.3, 4.5]
1 agent  obtenues  : [0.0, 1.0, 1.5, 2.3, 4.5]
2 agents attendues : [0.0, 0.0, 0.0, 0.3, 0.5]
2 agents obtenues  : [0.0, 0.0, 0.0, 0.3, 0.5]
agents attendus    : [1, 2, 1, 2, 1]
agents obtenus     : [1, 2, 1, 2, 1]


**Task 2. Add two tests.** Add one limiting case whose answer is obvious, such as enough agents for every caller to start immediately. Add one invalid input and state which exception should be raised. Keep both tests in the notebook.


In [11]:
def assert_raises(expected_exception, function, *args):
    """Réussit seulement si function(*args) lève exactement cette exception."""
    try:
        function(*args)
    except expected_exception:
        return
    except Exception as error:
        raise AssertionError(f"attendu {expected_exception.__name__}, obtenu {type(error).__name__}")
    raise AssertionError(f"{expected_exception.__name__} non levée")

def test_enough_agents_means_no_waiting():
    # Cas limite : autant d'agents que de clients -> personne n'attend.
    n = len(hand_arrivals)
    records = simulate_queue(hand_arrivals, hand_services, n)
    print("  attentes avec", n, "agents :", [r["wait"] for r in records])
    print("  agents utilisés         :", [r["agent"] for r in records])
    assert_close_list([r["wait"] for r in records], [0.0] * n)
    assert_close_list([r["start"] for r in records], hand_arrivals)
    assert [r["agent"] for r in records] == list(range(1, n + 1))

def test_zero_agents_raises_value_error():
    # Entrée invalide : 0 agent -> ValueError.
    try:
        simulate_queue(hand_arrivals, hand_services, 0)
    except ValueError as error:
        print("  exception levée :", type(error).__name__, "-", error)
        return
    raise AssertionError("ValueError attendue, rien n'a été levé")

test_enough_agents_means_no_waiting();   print("test limite OK")
test_zero_agents_raises_value_error();   print("test invalide OK")

  attentes avec 5 agents : [0.0, 0.0, 0.0, 0.0, 0.0]
  agents utilisés         : [1, 2, 3, 4, 5]
test limite OK
  exception levée : ValueError - number_of_agents must be a positive integer
test invalide OK


## 3. Reproduce the morning arrival model

The lecture selected ten regular working days (Sunday–Thursday) from 1–14 November as training data and kept twelve later regular working days for a check. The seven time periods were chosen from the training profile and are provisional.

Before running the code, predict:

1. How many 15-minute counts belong to the 09:00–11:00 morning period on ten days?
2. Why should Friday and Saturday not be pooled with the regular working days?


In [12]:
filename = "anonymousbank_nov1999_arrivals_15min.csv"
candidates = [Path.cwd() / "data" / filename, Path.cwd().parent / "data" / filename]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Start from the repository root or from notebooks/.")

def minutes_from_midnight(clock):
    hour, minute = map(int, clock.split(":"))
    return 60 * hour + minute

with data_path.open(newline="", encoding="utf-8") as handle:
    rows = [
        {
            "date": row["date"],
            "time": row["time"],
            "minute": minutes_from_midnight(row["time"]),
            "entries": int(row["entries"]),
        }
        for row in csv.DictReader(handle)
    ]

assert len(rows) == 30 * 96
assert sum(row["entries"] for row in rows) == 40_722
print(f"Loaded {len(rows):,} quarter-hour counts from {data_path}")


Loaded 2,880 quarter-hour counts from g:\Mon Drive\Documents GD\M2\RENF INFO 1 MULLER\stochastic-modeling-simulation-ds\data\anonymousbank_nov1999_arrivals_15min.csv


In [13]:
print("Premières lignes :")
for row in rows[:3]: print(row)
print("Nombre de jours :", len({r["date"] for r in rows}))

Premières lignes :
{'date': '1999-11-01', 'time': '00:00', 'minute': 0, 'entries': 5}
{'date': '1999-11-01', 'time': '00:15', 'minute': 15, 'entries': 2}
{'date': '1999-11-01', 'time': '00:30', 'minute': 30, 'entries': 0}
Nombre de jours : 30


In [14]:
# Python numbers Monday as 0, ..., Friday as 4, Saturday as 5, Sunday as 6.
regular_dates = sorted(
    {
        row["date"]
        for row in rows
        if datetime.strptime(row["date"], "%Y-%m-%d").weekday() not in (4, 5)
    }
)
training_dates = regular_dates[:10]
heldout_dates = regular_dates[10:]

periods = [
    ("Night", 0, 7 * 60),
    ("Opening", 7 * 60, 8 * 60),
    ("Build-up", 8 * 60, 9 * 60),
    ("Morning peak", 9 * 60, 11 * 60),
    ("Daytime", 11 * 60, 17 * 60),
    ("Evening", 17 * 60, 23 * 60),
    ("Late", 23 * 60, 24 * 60),
]

def period_name(minute):
    return next(name for name, start, end in periods if start <= minute < end)

def counts_by_period(selected_dates):
    result = {name: [] for name, _, _ in periods}
    for row in rows:
        if row["date"] in selected_dates:
            result[period_name(row["minute"])].append(row["entries"])
    return result

training_counts = counts_by_period(training_dates)
heldout_counts = counts_by_period(heldout_dates)
fitted_means = {
    name: statistics.mean(values)
    for name, values in training_counts.items()
}

print("Training dates:", ", ".join(training_dates))
print("Held-out dates:", ", ".join(heldout_dates))
print()
print(f"{'Period':<15} {'Training counts':>15} {'Fitted mean':>13}")
for name, _, _ in periods:
    print(f"{name:<15} {len(training_counts[name]):>15} {fitted_means[name]:>13.1f}")


Training dates: 1999-11-01, 1999-11-02, 1999-11-03, 1999-11-04, 1999-11-07, 1999-11-08, 1999-11-09, 1999-11-10, 1999-11-11, 1999-11-14
Held-out dates: 1999-11-15, 1999-11-16, 1999-11-17, 1999-11-18, 1999-11-21, 1999-11-22, 1999-11-23, 1999-11-24, 1999-11-25, 1999-11-28, 1999-11-29, 1999-11-30

Period          Training counts   Fitted mean
Night                       280           0.8
Opening                      40          11.3
Build-up                     40          23.0
Morning peak                 80          39.9
Daytime                     240          32.6
Evening                     240          19.6
Late                         40          12.7


**Task 3.** Explain what one value in `training_counts["Morning peak"]` represents. Why are there 80 values? Explain why the fitted mean is an estimate for a 15-minute count rather than for a complete morning.


In [15]:
first_day = training_dates[0]
day_values = [r["entries"] for r in rows
              if r["date"] == first_day and 9*60 <= r["minute"] < 11*60]
print("Jour", first_day, ": 8 créneaux ->", day_values)
print("Nombre de valeurs Morning peak :", len(training_counts["Morning peak"]), "= 8 x", len(training_dates))
m = fitted_means["Morning peak"]
print(f"Moyenne par créneau : {m:.2f} | par matinée (x8) : {8*m:.1f} | taux/min : {m/15:.3f}")

Jour 1999-11-01 : 8 créneaux -> [40, 56, 49, 51, 49, 51, 53, 45]
Nombre de valeurs Morning peak : 80 = 8 x 10
Moyenne par créneau : 39.85 | par matinée (x8) : 318.8 | taux/min : 2.657


### From a fixed profile to variable days

**Candidate 1** uses one fixed mean for every 15-minute slot in a period:

$$
N_{d,s} \sim \mathrm{Poisson}\!\left(\widehat{m}_{b(s)}\right).
$$

It therefore predicts that the variance of a slot count equals its mean. The held-out counts vary more than that during the busy periods.

**Candidate 2** draws one multiplier for the whole day, with $\mathbb{E}[G_d]=1$ and $\operatorname{Var}(G_d)=\alpha$, and then uses

$$
N_{d,s} \mid G_d
\sim \mathrm{Poisson}\!\left(G_d\widehat{m}_{b(s)}\right).
$$

The same $G_d$ affects all 96 slots, so a generally busy day remains busy across periods. We estimate $\alpha$ from the ten complete daily totals, then choose a Gamma distribution with the fitted mean and variance so that we can simulate a positive multiplier.


In [16]:
daily_totals = [
    sum(row["entries"] for row in rows if row["date"] == day)
    for day in training_dates
]
daily_mean = statistics.mean(daily_totals)
daily_variance = statistics.variance(daily_totals)

# Candidate 2 assumes E[G_d] = 1 and Var(G_d) = alpha.
alpha = (daily_variance - daily_mean) / daily_mean**2
gamma_shape = 1 / alpha
gamma_scale = alpha

print("Training daily totals:", daily_totals)
print(f"Mean daily total: {daily_mean:.1f}")
print(f"Sample variance of daily totals: {daily_variance:.1f}")
print(f"Estimated Var(G_d) = alpha: {alpha:.5f}")
print(f"Gamma shape: {gamma_shape:.1f}; Gamma scale: {gamma_scale:.5f}")


Training daily totals: [2011, 2200, 1705, 1504, 1396, 1555, 1879, 1877, 1860, 1818]
Mean daily total: 1780.5
Sample variance of daily totals: 59632.7
Estimated Var(G_d) = alpha: 0.01825
Gamma shape: 54.8; Gamma scale: 0.01825


In [17]:
print(
    f"{'Period':<15} {'Model mean':>11} {'Held-out mean':>14} "
    f"{'Model variance':>15} {'Held-out variance':>18}"
)
for name, _, _ in periods:
    model_mean = fitted_means[name]
    # Var(N) = E[Var(N|G)] + Var(E[N|G]) = m + alpha*m^2.
    model_variance = model_mean + alpha * model_mean**2
    observed_mean = statistics.mean(heldout_counts[name])
    observed_variance = statistics.variance(heldout_counts[name])
    print(
        f"{name:<15} {model_mean:>11.1f} {observed_mean:>14.1f} "
        f"{model_variance:>15.1f} {observed_variance:>18.1f}"
    )


Period           Model mean  Held-out mean  Model variance  Held-out variance
Night                   0.8            0.5             0.8                0.8
Opening                11.3            8.2            13.6                9.2
Build-up               23.0           19.4            32.6               42.7
Morning peak           39.9           38.6            68.8               67.8
Daytime                32.6           31.0            51.9               55.0
Evening                19.6           18.6            26.6               73.8
Late                   12.7           11.6            15.6               21.8


**Task 4.** For the morning peak and daytime periods, compare model and held-out means and variances. What improved relative to a fixed-rate Poisson model? Identify one period where Candidate 2 still fails. Write a two-sentence conclusion stating why the model is usable for a provisional morning experiment and why it is not accepted for the complete day.


In [18]:
def compare_models(names):
    print(f"{'Période':<14}{'Moy. test':>10}{'Moy. modèle':>12}{'Var Poisson':>12}{'Var cand. 2':>12}{'Var test':>10}")
    for name in names:
        m = fitted_means[name]
        print(f"{name:<14}{statistics.mean(heldout_counts[name]):>10.1f}{m:>12.1f}"
              f"{m:>12.1f}{m + alpha*m**2:>12.1f}{statistics.variance(heldout_counts[name]):>10.1f}")

compare_models(["Morning peak", "Daytime"])

print("\nVar cand. 2 / Var test, toutes périodes (1.00 = parfait) :")
for name, _, _ in periods:
    m = fitted_means[name]
    print(f"  {name:<14}{(m + alpha*m**2) / statistics.variance(heldout_counts[name]):>6.2f}")

Période        Moy. test Moy. modèle Var Poisson Var cand. 2  Var test
Morning peak        38.6        39.9        39.9        68.8      67.8
Daytime             31.0        32.6        32.6        51.9      55.0

Var cand. 2 / Var test, toutes périodes (1.00 = parfait) :
  Night           1.08
  Opening         1.48
  Build-up        0.76
  Morning peak    1.01
  Daytime         0.94
  Evening         0.36
  Late            0.72


## 4. Summarize customers and the system over time

Waiting times form a distribution **across customers**. Queue length and the number of busy agents form distributions **over elapsed time**. A state lasting ten minutes must receive ten times the weight of a state lasting one minute.

The helper below reconstructs intervals between state changes. At each interval midpoint it counts callers waiting and agents busy, then weights that state by the interval duration.


In [19]:
def linear_quantile(values, probability):
    ordered = sorted(values)
    if not ordered:
        return math.nan
    position = (len(ordered) - 1) * probability
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

def weighted_quantile(distribution, probability):
    cumulative = 0.0
    for value in sorted(distribution):
        cumulative += distribution[value]
        if cumulative >= probability:
            return value
    return max(distribution, default=0)

def state_distributions(records, horizon):
    boundaries = {0.0, horizon}
    for record in records:
        for key in ("arrival", "start", "departure"):
            if 0.0 < record[key] < horizon:
                boundaries.add(record[key])

    queue_duration = defaultdict(float)
    busy_duration = defaultdict(float)
    ordered = sorted(boundaries)
    for left, right in zip(ordered, ordered[1:]):
        midpoint = (left + right) / 2
        queue = sum(
            record["arrival"] <= midpoint < record["start"]
            for record in records
        )
        busy = sum(
            record["start"] <= midpoint < record["departure"]
            for record in records
        )
        queue_duration[queue] += right - left
        busy_duration[busy] += right - left

    queue_distribution = {
        state: duration / horizon
        for state, duration in queue_duration.items()
    }
    busy_distribution = {
        state: duration / horizon
        for state, duration in busy_duration.items()
    }
    return queue_distribution, busy_distribution

def summarize_queue(records, number_of_agents, horizon):
    waits = [record["wait"] for record in records]
    queue_distribution, busy_distribution = state_distributions(records, horizon)
    mean_queue = sum(
        state * probability
        for state, probability in queue_distribution.items()
    )
    mean_busy = sum(
        state * probability
        for state, probability in busy_distribution.items()
    )
    return {
        "calls": len(records),
        "mean_wait": statistics.mean(waits) if waits else 0.0,
        "median_wait": statistics.median(waits) if waits else 0.0,
        "p90_wait": linear_quantile(waits, 0.90),
        "fraction_wait": (
            sum(wait > 1e-12 for wait in waits) / len(waits) if waits else 0.0
        ),
        "mean_queue": mean_queue,
        "p90_queue": weighted_quantile(queue_distribution, 0.90),
        "utilization": mean_busy / number_of_agents,
        "all_busy": busy_distribution.get(number_of_agents, 0.0),
        "unfinished": sum(
            record["departure"] > horizon for record in records
        ),
    }


In [20]:
two_agent_records = simulate_queue(hand_arrivals, hand_services, 2)
queue_distribution, busy_distribution = state_distributions(
    two_agent_records, horizon=9.0
)
print("Fraction of time at each queue length:", dict(sorted(queue_distribution.items())))
print("Fraction of time with each number of busy agents:", dict(sorted(busy_distribution.items())))
print("Sums:", sum(queue_distribution.values()), sum(busy_distribution.values()))


Fraction of time at each queue length: {0: 0.911111111111111, 1: 0.08888888888888886}
Fraction of time with each number of busy agents: {0: 0.3888888888888889, 1: 0.2222222222222222, 2: 0.3888888888888889}
Sums: 0.9999999999999999 1.0


In [21]:
records = simulate_queue(hand_arrivals, hand_services, 2)
horizon = 9.0

def print_intervals(records, horizon):
    boundaries = {0.0, horizon}
    for r in records:
        for key in ("arrival", "start", "departure"):
            if 0.0 < r[key] < horizon:
                boundaries.add(r[key])
    ordered = sorted(boundaries)
    print(f"{'Intervalle':>14}{'Durée':>7}{'File':>6}{'Occupés':>9}")
    for left, right in zip(ordered, ordered[1:]):
        mid = (left + right) / 2
        queue = sum(r["arrival"] <= mid < r["start"] for r in records)
        busy = sum(r["start"] <= mid < r["departure"] for r in records)
        print(f"[{left:>4.1f}, {right:>4.1f}){right - left:>7.1f}{queue:>6}{busy:>9}")
    return ordered

ordered = print_intervals(records, horizon)

    Intervalle  Durée  File  Occupés
[ 0.0,  1.0)    1.0     0        1
[ 1.0,  2.0)    1.0     0        2
[ 2.0,  2.5)    0.5     0        1
[ 2.5,  2.7)    0.2     0        2
[ 2.7,  3.0)    0.3     1        2
[ 3.0,  3.5)    0.5     1        2
[ 3.5,  5.0)    1.5     0        2
[ 5.0,  5.5)    0.5     0        1
[ 5.5,  9.0)    3.5     0        0


**Task 5.** Explain why the code weights states by elapsed time rather than counting how often a state appears at an event. For the hand trace, identify one interval during which both agents are busy and verify its duration manually.


### Task 5 — Réponse

**Pourquoi pondérer par la durée ?**
La file moyenne, l'utilisation et la « fraction du temps où tous les agents sont occupés » sont des **moyennes dans le temps** : elles demandent ce que le système fait à un instant pris au hasard. Compter les états aux événements sur-représenterait les périodes agitées (beaucoup de changements en peu de temps) et sous-représenterait les périodes calmes (peu de changements, mais qui durent).

Sur la trace à la main (2 agents, horizon 9 min), le tableau le montre pour l'état « 0 agent occupé » :

| Méthode | Part de l'état « 0 occupé » |
|---|---|
| Pondérée par la durée (3.5 min sur 9) | **39 %** |
| Comptée une fois par intervalle (1 intervalle sur 9) | **11 %** |

**Vérification à la main d'un intervalle où les deux agents sont occupés : [3.5, 5.0).**

- Le client 4 est servi par l'agent 2 de 3.0 à 5.5.
- Le client 5 est servi par l'agent 1 de 3.5 à 5.0.
- Le client 3 quitte l'agent 1 à 3.5, et le client 5 le quitte à 5.0.

Entre 3.5 et 5.0, les deux agents sont donc occupés et personne n'attend. Durée : 5.0 − 3.5 = **1.5 minute**, ce qui correspond à la ligne du tableau d'intervalles.

**Temps total avec 2 agents occupés :**

| Intervalle | Durée |
|---|---|
| [1.0, 2.0) | 1.0 |
| [2.5, 3.5) | 1.0 |
| [3.5, 5.0) | 1.5 |
| **Total** | **3.5 min** |

Soit 3.5 / 9 ≈ **0.389** de l'horizon, la valeur de `all_busy` dans le résumé.

## 5. Generate one 09:00–11:00 arrival path

We compare two generators.

**Parametric generator.** Draw one morning multiplier $G_d$ from the fitted Gamma distribution. The morning rate is

$$
\lambda_d = \frac{G_d\widehat m_{\mathrm{morning}}}{15}
$$

calls per minute. Draw the time to the next arrival from an exponential distribution with rate $\lambda_d$, add it to the clock, and repeat until 120 minutes.

**Whole-morning bootstrap.** Select one of the ten ordered vectors of eight morning counts with replacement. Keep the intervals in clock order. Conditional on each count, place that many arrivals independently and uniformly inside its 15-minute interval. Selecting the profile is the bootstrap; uniform placement is an additional assumption because the extract omits exact event times.


In [22]:
morning_profiles = {
    day: sorted(
        [
            row for row in rows
            if row["date"] == day and 9 * 60 <= row["minute"] < 11 * 60
        ],
        key=lambda row: row["minute"],
    )
    for day in training_dates
}

def generate_parametric_morning(rng, fitted_mean, alpha, horizon=120.0):
    multiplier = rng.gammavariate(1 / alpha, alpha)
    rate_per_minute = multiplier * fitted_mean / 15
    arrivals = []
    clock = 0.0
    while True:
        clock += rng.expovariate(rate_per_minute)
        if clock >= horizon:
            break
        arrivals.append(clock)
    return arrivals, multiplier

def generate_bootstrap_morning(rng, profiles):
    selected_date = rng.choice(sorted(profiles))
    arrivals = []
    for row in profiles[selected_date]:
        interval_start = row["minute"] - 9 * 60
        arrivals.extend(
            interval_start + 15 * rng.random()
            for _ in range(row["entries"])
        )
    return sorted(arrivals), selected_date

parametric_arrivals, multiplier = generate_parametric_morning(
    random.Random(202601),
    fitted_means["Morning peak"],
    alpha,
)
bootstrap_arrivals, selected_date = generate_bootstrap_morning(
    random.Random(202602),
    morning_profiles,
)

print(
    f"Parametric path: {len(parametric_arrivals)} calls; "
    f"G_d = {multiplier:.3f}"
)
print(
    f"Bootstrap path: {len(bootstrap_arrivals)} calls; "
    f"selected date = {selected_date}"
)


Parametric path: 382 calls; G_d = 1.158
Bootstrap path: 314 calls; selected date = 1999-11-10


In [23]:
def generate_services(number_of_calls, mean_service, rng):
    rate_per_minute = 1 / mean_service
    return [
        rng.expovariate(rate_per_minute)
        for _ in range(number_of_calls)
    ]

mean_service = 6.0
parametric_services = generate_services(
    len(parametric_arrivals), mean_service, random.Random(202603)
)
bootstrap_services = generate_services(
    len(bootstrap_arrivals), mean_service, random.Random(202604)
)

print(f"Scenario service mean: {mean_service:.1f} minutes")
print(f"Parametric-path sample mean: {statistics.mean(parametric_services):.2f}")
print(f"Bootstrap-path sample mean: {statistics.mean(bootstrap_services):.2f}")


Scenario service mean: 6.0 minutes
Parametric-path sample mean: 5.71
Bootstrap-path sample mean: 5.83


**Task 6.** State which parts of the two arrival paths come from observed data, which come from a fitted model, and which come from an additional simulation assumption. Why must the eight bootstrap intervals remain in clock order? Why are the two sample service means not exactly 6 minutes?


### Task 6 — Réponse

**1. Ce qui vient des données, du modèle ajusté, et d'une hypothèse de simulation**

| | Générateur paramétrique | Bootstrap de matinées entières |
|---|---|---|
| **Données observées** | Seulement à travers des *estimations* : la moyenne `fitted_means["Morning peak"]` (≈ 39.9 par créneau, calculée sur 80 comptes observés) et $\widehat\alpha$ (estimé à partir de 10 totaux journaliers observés). | Les **8 comptes réels** d'une matinée d'entraînement, dans leur ordre réel. |
| **Modèle ajusté** | Le multiplicateur de jour $G_d$ (loi Gamma, moyenne 1, variance $\widehat\alpha$) et des arrivées de Poisson, c'est-à-dire des écarts exponentiels de taux $G_d\,\widehat m/15$. | Presque aucun : le « modèle » est la distribution empirique des 10 matinées d'entraînement, chacune tirée avec la probabilité 1/10. |
| **Hypothèse de simulation ajoutée** | Le taux est **constant** sur les deux heures (aucune forme à l'intérieur de la matinée). | Les arrivées sont placées **uniformément** dans chaque créneau de 15 min, car le fichier ne contient pas les heures exactes. |

Dans les deux cas, les **durées de service ne sont jamais observées** : elles sont exponentielles de moyenne 6 min par hypothèse du scénario (le fichier ne compte que des entrées au VRU).

Autre limite du bootstrap : il ne peut produire que 10 matinées possibles, donc aucune matinée plus chargée que la plus chargée des 10 observées.

**2. Pourquoi les 8 intervalles doivent rester dans l'ordre chronologique**

L'ordre porte de l'information. Un 10h00–10h30 très chargé suivi d'un 10h30–11h00 calme donne une file différente des mêmes comptes mélangés, car l'attente dépend de *quand* les arrivées se regroupent par rapport à la capacité des agents. Garder la matinée entière conserve aussi la dépendance entre créneaux d'un même jour (un jour chargé l'est toute la matinée), qui est justement ce que le candidat 2 cherche à capturer. Mélanger les créneaux détruirait cette structure.

**3. Pourquoi les deux moyennes de service ne valent pas exactement 6 minutes**

Chaque moyenne est calculée sur un nombre fini $n$ de durées tirées au hasard. Son écart-type vaut $6/\sqrt{n}$, soit environ 0.34 minute pour $n \approx 320$ appels. Seule la moyenne **théorique** est égale à 6. Les deux échantillons diffèrent parce qu'ils n'ont pas le même nombre d'appels et pas la même graine (202603 et 202604).

Dans mon exécution : moyenne paramétrique = **`___`**, moyenne bootstrap = **`___`** (à remplir avec les sorties de la cellule 23).

## 6. Use the theoretical model to choose staffing levels

The theoretical reference is an infinite-horizon $M/M/c$ queue: constant-rate Poisson arrivals, independent exponential service times, $c$ identical agents, FIFO service, unlimited waiting room, and no abandonment.

When a queue is present, all agents are busy. Their combined service capacity is $c\mu$ calls per minute. The model has a stationary distribution exactly when

$$
\rho = \frac{\lambda}{c\mu} < 1.
$$

This is a capacity screen. It does not predict the finite-morning waiting-time distribution.


In [24]:
arrival_rate = fitted_means["Morning peak"] / 15
service_rate = 1 / mean_service

print(f"Arrival rate on an average-demand morning: {arrival_rate:.3f} calls/min")
print(f"Service rate per agent: {service_rate:.3f} calls/min")
print(f"Offered load lambda/mu: {arrival_rate / service_rate:.2f} agents")
print()
print(f"{'Agents':>6} {'Capacity':>10} {'rho':>8} {'Stationary?':>13}")
for number_of_agents in (15, 16, 17):
    capacity = number_of_agents * service_rate
    rho = arrival_rate / capacity
    print(
        f"{number_of_agents:>6} {capacity:>10.3f} {rho:>8.3f} "
        f"{str(rho < 1):>13}"
    )


Arrival rate on an average-demand morning: 2.657 calls/min
Service rate per agent: 0.167 calls/min
Offered load lambda/mu: 15.94 agents

Agents   Capacity      rho   Stationary?
    15      2.500    1.063         False
    16      2.667    0.996          True
    17      2.833    0.938          True


In [25]:
arrival_rate = fitted_means["Morning peak"] / 15
service_rate = 1 / mean_service
offered_load = arrival_rate / service_rate

print(f"lambda = {arrival_rate:.3f} appels/min | mu = {service_rate:.3f} /min/agent")
print(f"Charge offerte lambda/mu = {offered_load:.2f} agents")
print()
print(f"{'Agents':>6} {'Capacité c*mu':>14} {'rho':>8} {'Stable ?':>9} {'Marge (agents)':>15}")
for c in (15, 16, 17):
    rho = arrival_rate / (c * service_rate)
    print(f"{c:>6} {c * service_rate:>14.3f} {rho:>8.4f} {str(rho < 1):>9} {c - offered_load:>15.2f}")

lambda = 2.657 appels/min | mu = 0.167 /min/agent
Charge offerte lambda/mu = 15.94 agents

Agents  Capacité c*mu      rho  Stable ?  Marge (agents)
    15          2.500   1.0627     False           -0.94
    16          2.667   0.9963      True            0.06
    17          2.833   0.9376      True            1.06


In [26]:
print(f"{'G_d':>6}" + "".join(f"{'c=' + str(c):>9}" for c in (15, 16, 17)))
for g in (0.9, 1.0, 1.05, 1.1, 1.2):
    print(f"{g:>6.2f}" + "".join(f"{g * offered_load / c:>9.3f}" for c in (15, 16, 17)))
print()
for c in (16, 17):
    print(f"c = {c} : le test de capacité échoue dès que G_d > {c / offered_load:.3f}")

   G_d     c=15     c=16     c=17
  0.90    0.956    0.897    0.844
  1.00    1.063    0.996    0.938
  1.05    1.116    1.046    0.985
  1.10    1.169    1.096    1.031
  1.20    1.275    1.196    1.125

c = 16 : le test de capacité échoue dès que G_d > 1.004
c = 17 : le test de capacité échoue dès que G_d > 1.066


**Task 7.** Explain why 16 is the smallest stable integer for the average-demand reference model and why 17 is the next useful comparison. If a simulated morning has multiplier `G_d > 1`, how does that change `rho`? Explain why `rho < 1` does not imply short waits.


### Task 7 — Réponse

**Données de départ.** La moyenne ajustée du Morning peak est d'environ 39.9 appels par créneau de 15 min, donc $\lambda \approx 39.9/15 \approx 2.66$ appels/min. Le service moyen est de 6 min, donc $\mu = 1/6$ appel/min par agent. La charge offerte vaut $\lambda/\mu \approx 15.96$ agents.

| Agents $c$ | Capacité $c\mu$ | $\rho = \lambda/(c\mu)$ | Stationnaire ? |
|---|---|---|---|
| 15 | 2.500 | ≈ 1.064 | non |
| 16 | 2.667 | ≈ 0.998 | oui, de justesse |
| 17 | 2.833 | ≈ 0.939 | oui, avec une marge |

**1. Pourquoi 16 est le plus petit entier stable.**
Quand une file existe, tous les agents sont occupés et leur capacité combinée est $c\mu$. Le système admet une distribution stationnaire exactement quand $\rho = \lambda/(c\mu) < 1$, c'est-à-dire $c > \lambda/\mu \approx 15.96$. Avec 15 agents, $\rho \approx 1.064 \ge 1$ : il arrive plus de travail que la capacité n'en traite, et la file n'est pas stable. Le plus petit entier supérieur à 15.96 est **16**.

**2. Pourquoi 17 est la comparaison suivante utile.**
16 agents n'ont que 0.04 agent de marge par rapport à la charge offerte ($\rho \approx 0.998$) : on est presque sur la frontière. 17 agents montrent ce qu'apporte **un agent de plus en partant du bord** : $\rho$ passe de 0.998 à 0.939 et la marge à environ 1 agent. C'est la comparaison qui donne le plus d'information sur la zone critique.

**3. Effet d'un multiplicateur $G_d > 1$ sur $\rho$.**
Le multiplicateur de jour multiplie le taux d'arrivée, donc $\rho_d = G_d \cdot \rho$. Le test de capacité échoue dès que $G_d > c\mu/\lambda = c/(\lambda/\mu)$ :

- avec 16 agents, dès $G_d > 16/15.96 \approx 1.003$, soit un jour seulement 0.3 % plus chargé que la moyenne ;
- avec 17 agents, dès $G_d > 17/15.96 \approx 1.065$, soit un jour 6.5 % plus chargé.

Comme $G_d$ varie d'un jour à l'autre (écart-type $\sqrt{\widehat\alpha}$), 16 agents sont stables sur un jour *exactement moyen* mais pas sur un jour typique tiré au hasard : un jour sur deux environ est trop chargé pour eux.

**4. Pourquoi $\rho < 1$ n'implique pas des attentes courtes.**

- $\rho < 1$ garantit seulement que la file ne croît pas sans limite *à long terme*. Près de $\rho = 1$, l'attente stationnaire est très grande et très sensible. La formule d'Erlang C donne, avec ces valeurs, une attente moyenne d'environ **148 min** à 16 agents contre environ **4 min** à 17 agents, alors que $\rho$ passe seulement de 0.998 à 0.939. Ces chiffres sont très sensibles à l'arrondi de la moyenne : avec 39.8 au lieu de 39.9, on tombe à 73 min environ, et avec 40.0 le système devient instable.
- Le modèle M/M/c décrit un **horizon infini** avec un taux **constant**. Notre matinée dure 2 heures, **commence à vide** et est aléatoire : la file n'a pas le temps d'atteindre le régime stationnaire, et une matinée plus chargée que la moyenne peut saturer les agents même si la charge *moyenne* est inférieure à leur capacité.

Le calcul de $\rho$ est donc un **filtre de capacité** : il écarte les effectifs clairement insuffisants (ici 15) et signale les effectifs fragiles (16). Il ne prédit pas la distribution des temps d'attente d'une matinée finie : c'est le rôle de la simulation (section 7).

## 7. Simulate around the capacity boundary

Within each arrival generator, keep the arrivals and service requirements fixed while changing only the number of agents. This paired comparison isolates the staffing change for that repetition.

Every generated arrival occurs before 11:00. The queue schedule then continues until those callers finish service. Waiting-time summaries use this arrival cohort; queue-length and busy-agent summaries cover only 09:00–11:00; `Unfinished` counts callers still present at 11:00.


In [27]:
def print_staffing_scan(label, arrivals, services, horizon=120.0):
    print(label)
    print(
        f"{'Agents':>6} {'Calls':>6} {'Mean W':>8} {'Median W':>9} "
        f"{'P90 W':>8} {'Wait %':>8} {'Mean Q':>8} {'P90 Q':>7} "
        f"{'Util.':>7} {'All busy %':>11} {'Unfinished':>11}"
    )
    summaries = {}
    for number_of_agents in (15, 16, 17):
        records = simulate_queue(arrivals, services, number_of_agents)
        summary = summarize_queue(records, number_of_agents, horizon)
        summaries[number_of_agents] = summary
        print(
            f"{number_of_agents:>6} {summary['calls']:>6} "
            f"{summary['mean_wait']:>8.2f} {summary['median_wait']:>9.2f} "
            f"{summary['p90_wait']:>8.2f} {100 * summary['fraction_wait']:>7.1f}% "
            f"{summary['mean_queue']:>8.2f} {summary['p90_queue']:>7} "
            f"{summary['utilization']:>7.3f} {100 * summary['all_busy']:>10.1f}% "
            f"{summary['unfinished']:>11}"
        )
    return summaries

parametric_summaries = print_staffing_scan(
    "Parametric arrival path",
    parametric_arrivals,
    parametric_services,
)
print()
bootstrap_summaries = print_staffing_scan(
    "Whole-morning bootstrap path",
    bootstrap_arrivals,
    bootstrap_services,
)


Parametric arrival path
Agents  Calls   Mean W  Median W    P90 W   Wait %   Mean Q   P90 Q   Util.  All busy %  Unfinished
    15    382     8.15      5.95    20.38    82.7%    18.46      53   0.932       79.6%          83
    16    382     5.78      4.31    15.10    74.3%    13.78      41   0.912       70.0%          68
    17    382     4.15      2.91    10.91    64.4%    10.41      31   0.888       60.7%          62

Whole-morning bootstrap path
Agents  Calls   Mean W  Median W    P90 W   Wait %   Mean Q   P90 Q   Util.  All busy %  Unfinished
    15    314     5.64      5.69    11.69    79.3%    13.49      30   0.906       77.9%          48
    16    314     3.01      2.89     6.72    76.4%     7.74      17   0.894       74.4%          27
    17    314     1.49      1.15     3.82    61.1%     3.90      10   0.868       62.0%          15


**Task 8. Locate the transition.** For each generator, describe what changes from 15 to 16 to 17 agents. Use waiting time, queue length, all-busy time, and unfinished calls. Does the finite two-hour simulation show a sharp boundary at the same place as the infinite-horizon capacity test?

**Task 9. Compare 16 with 17.** For each generator, calculate the absolute and percentage reduction in mean wait and 90th-percentile wait. Do both arrival generators support the same qualitative conclusion? If they disagree in magnitude, what does that tell you?

Run the parametric experiment with one different seed. Record what changes, but do not tune the seed to obtain a preferred result. Week 3 will replace single-repetition comparisons with uncertainty estimates across many repetitions.


### Task 8 — Réponse : localiser la transition

**Remarque préalable sur les deux matinées.** Le chemin paramétrique contient 382 appels (≈ 1.2 fois la moyenne de 319), soit une charge d'environ 382 × 6 / 120 ≈ 19 agents ; le chemin bootstrap contient 314 appels, soit une charge d'environ 15.7 agents. Les deux matinées ne sont donc pas comparables en niveau, seulement en tendance.

**1. Ce qui change de 15 à 16 à 17 agents**

| Indicateur | Générateur | 15 agents | 16 agents | 17 agents |
|---|---|---|---|---|
| Attente moyenne (min) | Paramétrique | 8.15 | 5.78 | 4.15 |
| | Bootstrap | 5.64 | 3.01 | 1.49 |
| Attente P90 (min) | Paramétrique | 20.38 | 15.10 | 10.91 |
| | Bootstrap | 11.69 | 6.72 | 3.82 |
| Clients qui attendent (%) | Paramétrique | 82.7 | 74.3 | 64.4 |
| | Bootstrap | 79.3 | 76.4 | 61.1 |
| Longueur moyenne de la file | Paramétrique | 18.46 | 13.78 | 10.41 |
| | Bootstrap | 13.49 | 7.74 | 3.90 |
| File P90 | Paramétrique | 53 | 41 | 31 |
| | Bootstrap | 30 | 17 | 10 |
| Temps « tous occupés » (%) | Paramétrique | 79.6 | 70.0 | 60.7 |
| | Bootstrap | 77.9 | 74.4 | 62.0 |
| Appels non terminés à 11h | Paramétrique | 83 | 68 | 62 |
| | Bootstrap | 48 | 27 | 15 |

- **Temps d'attente.** Ils diminuent à chaque agent ajouté, dans les deux générateurs. En paramétrique, l'attente moyenne baisse d'environ 29 % de 15 à 16 puis de 28 % de 16 à 17 ; en bootstrap, d'environ 47 % puis 50 %. Le 90ᵉ centile suit la même tendance (par exemple 11.69 → 6.72 → 3.82 en bootstrap).
- **Part de clients qui attendent.** Elle baisse peu entre 15 et 16 (82.7 → 74.3 % en paramétrique, 79.3 → 76.4 % en bootstrap), puis nettement à 17 en bootstrap (61.1 %). Même à 17 agents, plus de 60 % des clients attendent un peu : l'attente est réduite, pas supprimée.
- **File.** La longueur moyenne et le P90 diminuent régulièrement : en bootstrap, la file moyenne passe de 13.49 à 7.74 puis 3.90.
- **Tous occupés et utilisation.** La part du temps où tous les agents sont occupés baisse avec l'effectif. L'utilisation reste **inférieure à 1 même à 15 agents** (0.93 et 0.91), alors que la charge dépasse la capacité : la matinée démarre à vide, et il faut quelques minutes pour que les agents se remplissent.
- **Appels non terminés à 11h.** Un arriéré subsiste dans tous les cas, et il diminue avec l'effectif : 83 → 68 → 62 en paramétrique, 48 → 27 → 15 en bootstrap. La baisse est la plus marquée en bootstrap.

**2. La simulation sur deux heures montre-t-elle une frontière nette au même endroit que le test de capacité ?**

**Non.** Le critère $\rho < 1$ est un interrupteur : instable à 15 ($\rho \approx 1.06$), stable à 16 ($\rho \approx 0.998$) sur un jour moyen. La simulation ne montre aucun saut entre 15 et 16 : chaque agent supplémentaire réduit l'attente et la file de façon **continue**, avec des gains en pourcentage assez réguliers (environ 28-29 % par agent en paramétrique, 47-50 % en bootstrap). Plusieurs raisons :

- la simulation dure 120 minutes et démarre à vide : la file n'a pas le temps d'atteindre le régime d'équilibre que décrit le modèle M/M/c ;
- chaque générateur ne fournit qu'**une seule matinée aléatoire** dont la charge réelle diffère de la charge moyenne (ici 19 agents pour la matinée paramétrique, 15.7 pour le bootstrap) ; sur la matinée paramétrique, même 17 agents sont en surcharge, il n'y a donc aucune « frontière » à franchir entre 15, 16 et 17 ;
- les attentes simulées sont de l'ordre de quelques minutes, très loin des environ 148 minutes que la formule d'Erlang C annonce à 16 agents en régime stationnaire : une matinée finie ne reproduit pas l'équilibre.

Le test de capacité reste utile comme filtre (il écarte 15 agents et signale la fragilité de 16), mais il ne localise pas à lui seul le changement de comportement d'une matinée particulière.

In [28]:
def reduction(summaries, key):
    before, after_ = summaries[16][key], summaries[17][key]
    absolute = before - after_
    percent = 100 * absolute / before if before > 0 else math.nan
    return before, after_, absolute, percent

print(f"{'Générateur':<12}{'Métrique':<10}{'16 agents':>10}{'17 agents':>10}{'Réd. abs.':>11}{'Réd. %':>9}")
for label, summaries in (("Paramétrique", parametric_summaries), ("Bootstrap", bootstrap_summaries)):
    for key, short in (("mean_wait", "Mean W"), ("p90_wait", "P90 W")):
        b, a_, d, p = reduction(summaries, key)
        print(f"{label:<12}{short:<10}{b:>10.2f}{a_:>10.2f}{d:>11.2f}{p:>8.1f}%")

# Une seule graine différente (ne PAS la choisir pour obtenir un résultat voulu).
alt_arrivals, alt_multiplier = generate_parametric_morning(random.Random(202605), fitted_means["Morning peak"], alpha)
alt_services = generate_services(len(alt_arrivals), mean_service, random.Random(202606))
print(f"\nAutre graine : {len(alt_arrivals)} appels, G_d = {alt_multiplier:.3f}")
alt_summaries = print_staffing_scan("Paramétrique, autre graine", alt_arrivals, alt_services)
for key, short in (("mean_wait", "Mean W"), ("p90_wait", "P90 W")):
    b, a_, d, p = reduction(alt_summaries, key)
    print(f"{short} : 16 -> 17 agents : {b:.2f} -> {a_:.2f} (réduction {d:.2f}, {p:.1f} %)")

Générateur  Métrique   16 agents 17 agents  Réd. abs.   Réd. %
ParamétriqueMean W          5.78      4.15       1.63    28.2%
ParamétriqueP90 W          15.10     10.91       4.19    27.7%
Bootstrap   Mean W          3.01      1.49       1.52    50.6%
Bootstrap   P90 W           6.72      3.82       2.90    43.2%

Autre graine : 341 appels, G_d = 1.036
Paramétrique, autre graine
Agents  Calls   Mean W  Median W    P90 W   Wait %   Mean Q   P90 Q   Util.  All busy %  Unfinished
    15    341     6.88      2.33    18.74    61.9%    14.84      41   0.866       58.7%          66
    16    341     5.32      1.91    14.21    58.1%    12.16      34   0.841       52.9%          57
    17    341     3.98      1.35    10.22    55.7%     9.57      26   0.819       50.4%          48
Mean W : 16 -> 17 agents : 5.32 -> 3.98 (réduction 1.35, 25.3 %)
P90 W : 16 -> 17 agents : 14.21 -> 10.22 (réduction 3.99, 28.1 %)


## 8. Use an LLM to design checks, then verify them

**Code-test prompt:**

> Design three minimal tests for this FIFO multi-agent simulator. Include simultaneous arrivals, a tie between free agents, and an invalid input. For valid cases, give every expected agent, start, wait, and departure; for the invalid case, state the expected error.

**Workflow-audit prompt:**

> Trace the claim “17 agents reduce waiting relative to 16” through the arrival data, service assumptions, random generator, queue code, and reported summaries. At each stage, propose one failure mode and one independent check.

Record:

1. the proposed case or claim;
2. your independent hand calculation or data check;
3. any correction to the LLM;
4. the useful test or workflow check you retained.

Add at least one verified case to the unit tests above. The value of the exercise is the retained check, even when the original code was correct.


## 9. Decide what this repetition supports

Write a short conclusion with four parts:

1. **Capacity:** what the $M/M/c$ calculation says about 15, 16, and 17 agents on an average-demand morning.
2. **Simulation:** what changed from 16 to 17 agents in this repetition, with at least two numerical summaries.
3. **Sensitivity:** whether the parametric and whole-morning bootstrap generators tell the same story.
4. **Boundary:** why this remains a classroom scenario rather than a bank staffing recommendation.

Before the exit quiz, confirm that your notebook contains the hand check, passing unit tests, the recorded seeds and selected bootstrap date, the staffing tables, and your conditional conclusion.
